# WhisperSubs ColabJapanese audio â†’ casual Indonesian SRT using kotoba-whisper-v2.1-ct2 (float16) + GPT-4o> **Before running:** Runtime â†’ Change runtime type â†’ T4 GPU>> **Workflow:** Run all cells top to bottom. Enter API key when prompted. Upload audio. Download SRT.

In [ ]:
# Cell 1: Install dependencies
!pip install -q faster-whisper openai
print('Dependencies installed.')

In [ ]:
# Cell 2: Configuration — run this once per session
import getpass
import os

OPENAI_API_KEY = getpass.getpass("OpenAI API Key: ")
GPT_MODEL = "gpt-4o"
WHISPER_MODEL = "jctv-tech/kotoba-whisper-v21-ct2"
BATCH_SIZE = 5

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"Config set.")
print(f"  GPT model   : {GPT_MODEL}")
print(f"  Whisper model: {WHISPER_MODEL}")
print(f"  Batch size  : {BATCH_SIZE} segments per GPT call")

In [ ]:
# Cell 3: Upload audio file
from google.colab import files

print("Select your audio file (MP3, WAV, M4A, FLAC, etc.):")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Re-run this cell and select a file.")

audio_filename = list(uploaded.keys())[0]
file_size_mb = len(uploaded[audio_filename]) / 1024 / 1024
print(f"
Ready: {audio_filename} ({file_size_mb:.1f} MB)")

In [ ]:
# Cell 4: Transcribe audio
# First run: downloads model to Colab (~3-5 min). Subsequent cells in same session use cached model.
from faster_whisper import WhisperModel

print(f"Loading model: {WHISPER_MODEL} on T4 GPU (float16)...")
print("(First run downloads ~3GB — takes 3-5 min. Cached for this session.)")

model = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")

print("Transcribing...")
segments_iter, info = model.transcribe(
    audio_filename,
    language="ja",
    beam_size=5,
    condition_on_previous_text=False,
    no_speech_threshold=0.6,
    compression_ratio_threshold=2.4,
    log_prob_threshold=-1.0,
    vad_filter=True,
    vad_parameters={
        "threshold": 0.5,
        "min_silence_duration_ms": 1000,
        "speech_pad_ms": 200,
    },
)

print(f"Detected language: {info.language} (confidence: {info.language_probability:.2f})")

segments = []
for seg in segments_iter:
    segments.append({
        "start": seg.start,
        "end": seg.end,
        "text": seg.text.strip(),
    })

print(f"\nTranscription complete! Total segments: {len(segments)}")
if segments:
    print(f"  First: [{segments[0]['start']:.2f}s → {segments[0]['end']:.2f}s] {segments[0]['text'][:80]}")
    print(f"  Last:  [{segments[-1]['start']:.2f}s → {segments[-1]['end']:.2f}s] {segments[-1]['text'][:80]}")

In [ ]:
# Cell 5: Translate segments to casual Indonesian via GPT-4o
import re
import time
from openai import OpenAI

TRANSLATION_SYSTEM_PROMPT = (
    "Kamu adalah penerjemah subtitle dari bahasa Jepang ke bahasa Indonesia. "
    "Terjemahkan setiap dialog dalam tanda [Dialog X] ke bahasa Indonesia.\n\n"
    "Aturan gaya bahasa:\n"
    "- Gunakan \"aku/kamu\" bukan \"saya/Anda\"\n"
    "- Pakai bahasa sehari-hari yang santai seperti ngobrol sama teman\n"
    "- Hindari bahasa baku/formal (jangan pakai \"telah\", \"namun\", \"dapat\", dll)\n"
    "- Hindari slang berat Jakarta (jangan pakai \"gue/lu\", \"anjir\", dll)\n"
    "- Singkat dan natural, seperti subtitle anime fansub\n"
    "- Jaga konteks antar dialog agar cerita tetap nyambung\n\n"
    "Pertahankan format [Dialog X] agar bisa dicocokkan kembali."
)

client = OpenAI(api_key=OPENAI_API_KEY)
translated_segments = []
total_batches = (len(segments) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_start in range(0, len(segments), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(segments))
    batch_segments = segments[batch_start:batch_end]
    batch_num = batch_start // BATCH_SIZE + 1

    print(f"Batch {batch_num}/{total_batches} — segments {batch_start + 1}-{batch_end}...")

    batch_texts = []
    for i, seg in enumerate(batch_segments):
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', seg['text'])
        batch_texts.append(f"[Dialog {i + 1}] {text}")

    combined_text = "\n".join(batch_texts)

    try:
        response = client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": TRANSLATION_SYSTEM_PROMPT},
                {"role": "user", "content": combined_text},
            ],
            temperature=0.6,
        )

        result_text = response.choices[0].message.content.strip()

        # Parse [Dialog N] markers out of GPT response
        translated_dialogs = {}
        current_dialog = None
        current_text = []

        for line in result_text.split('\n'):
            if line.strip().startswith('[Dialog'):
                if current_dialog is not None:
                    translated_dialogs[current_dialog] = ' '.join(current_text).strip()
                try:
                    dialog_num = int(line.split(']')[0].split()[-1])
                    current_dialog = dialog_num
                    text_after = line.split(']', 1)[1].strip() if ']' in line else ''
                    current_text = [text_after] if text_after else []
                except Exception:
                    continue
            elif current_dialog is not None and line.strip():
                current_text.append(line.strip())

        if current_dialog is not None:
            translated_dialogs[current_dialog] = ' '.join(current_text).strip()

        for i, seg in enumerate(batch_segments):
            translated = translated_dialogs.get(i + 1, '')
            if not translated:
                print(f"  Warning: Dialog {batch_start + i + 1} failed — using original JP text")
                translated = seg['text']
            translated_segments.append({
                'start': seg['start'],
                'end': seg['end'],
                'text': translated,
            })

        if batch_end < len(segments):
            time.sleep(1)

    except Exception as e:
        print(f"  Error in batch {batch_num}: {e}")
        for seg in batch_segments:
            translated_segments.append(seg)

print(f"\nTranslation complete! {len(translated_segments)} segments.")
if translated_segments:
    print(f"Sample translation: {translated_segments[0]['text']}")

In [ ]:
# Cell 6: Generate SRT file and download# Guard: ensure translations exist before generating SRTif 'translated_segments' not in dir() or not translated_segments:    raise RuntimeError("Run Cell 5 first to generate translations.")from google.colab import filesdef format_time(seconds):    total_ms = round(float(seconds) * 1000)    ms = total_ms % 1000    total_secs = total_ms // 1000    hours, remainder = divmod(total_secs, 3600)    minutes, secs = divmod(remainder, 60)    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"srt_content = ""for i, segment in enumerate(translated_segments):    start_time = format_time(segment['start'])    end_time = format_time(segment['end'])    srt_content += f"{i + 1}{start_time} --> {end_time}{segment['text'].strip()}"# Output filename derived from uploaded audio filenamebase_name = audio_filename.rsplit('.', 1)[0]output_filename = f"{base_name}_id.srt"with open(output_filename, 'w', encoding='utf-8') as f:    f.write(srt_content)print(f"SRT saved: {output_filename} ({len(translated_segments)} segments)")print("Sample (first 3 entries):")print(srt_content[:400])files.download(output_filename)print("Download triggered — check your browser downloads.")# Guard: ensure translations exist before generating SRTif 'translated_segments' not in dir() or not translated_segments:    raise RuntimeError("Run Cell 5 first to generate translations.")